The goal of this second lesson is to get familiar with [Superlinked](https://superlinked.com/) — a powerful framework for building multi-attribute vector search systems — which lets us handle complex, nuanced queries that go far beyond simple keyword matching or text-only embeddings.

> We'll use Superlinked to power the property search tool in our voice agent, enabling natural language queries like "I'm looking for a spacious apartment in downtown with at least 3 bedrooms under 500k"!

In this lesson, we'll teach you the fundamentals of Superlinked, starting with why traditional vector search falls short, and progressively building a complete property search system that combines semantic understanding with structured filters.

![Superlinked Logo](../static/superlinked_logo.png)


## Understanding Superlinked core concepts


Traditional vector search works great for matching text to text. But real-world queries are rarely just about text — they mix semantic meaning with numerical constraints and categorical filters.

Think about our real estate use case. When someone says *"spacious modern apartment under €300,000"*, they're expressing:
- **Text preferences**: "modern", "spacious" (semantic meaning)
- **Numerical constraints**: price under €300k, size preferences
- **Categorical filters**: location, property type

If you try to handle this with a standard text embedding, you run into problems. The embedding captures semantic similarity of text, but it doesn't understand that €301,000 is almost the same as €299,000, or that 95 square meters is close to 100 square meters.

---

### Why Not Just Use Common Alternatives?

You might be thinking: "Can't I solve this with metadata filters or multiple searches?" Let's look at the most common approaches and why they fall short:

**1. The "Stringify Everything" Approach**

The simplest idea: convert all your data to text before embedding it.

```
"3 bedroom apartment, 120 sqm, €450,000, Barcelona, Eixample district"
```

The problem? You lose the mathematical properties of numbers. The embedding doesn't understand that €449,000 is closer to €450,000 than €200,000 is. It just sees tokens like "four", "hundred", "fifty" — which may have no meaningful relationship in embedding space.

**2. Metadata Filters**

Another common approach: run a semantic search on the text, then apply hard filters on metadata.

```python
results = vector_db.search("modern apartment")
filtered = [r for r in results if r.price < 400000]
```

The issue here is that you're modeling a smooth preference with a binary step function. Is a €401,000 apartment really unacceptable if the user said "under €400k"? And what if filtering removes most results, leaving you with poor matches?

**3. Multiple Searches + Result Fusion**

You could run separate searches for each attribute (text, price, size) and then fuse the results using some ranking algorithm.

This is complex to implement and has a fundamental limitation: it doesn't capture how attributes *interact*. A user searching for "affordable spacious apartment" wants the best *combination* — not the most affordable apartment plus the most spacious one.

**4. Re-ranking**

Re-ranking involves getting a broad set of candidates first, then applying a more sophisticated model to re-order them.

This only works if the relevant items make it into your initial candidate set. For broad multi-attribute queries, the items you actually want might not score high enough on text similarity alone to survive the initial retrieval.

---

### How Superlinked Solves This

[Superlinked](https://superlinked.com/) takes a different approach: encode all your attributes into the **same vector space** using specialized encoders, and perform a single unified search.

It provides different **Space** types for different data:

- `TextSimilaritySpace`: For semantic understanding of text descriptions
- `NumberSpace`: For numerical attributes like price, size, ratings (with MINIMUM/MAXIMUM modes)
- `CategorySpace`: For categorical data like location, type
- `RecencySpace`: For time-based relevance

Each space knows how to properly encode its data type. Then, at query time, you can **weight each space** to prioritize what matters most — want to emphasize price? Bump up the price weight. Looking for something spacious? Increase the size weight.

This is perfect for voice agents! The LLM can dynamically adjust weights based on what the user emphasizes in their query.


---
**⚠️  IMPORTANT!  ⚠️**

Before moving forward, make sure you have:

1. Set up your `.env` file with your API keys
2. Installed the project dependencies
3. **Created a free Qdrant Cloud cluster** and added your `QDRANT__API_KEY` and `QDRANT__HOST` to `.env`

You can create a free Qdrant cluster at [cloud.qdrant.io](https://cloud.qdrant.io). The free tier is more than enough for this course!

Your `.env` file should include:
```
GROQ__API_KEY=your_groq_api_key
QDRANT__API_KEY=your_qdrant_api_key
QDRANT__HOST=your-cluster-id.aws.cloud.qdrant.io
QDRANT__PORT=6333
QDRANT__USE_HTTPS=true
```

💡 **Note on Qdrant deployment**: Since Qdrant is open source, in our actual code implementation we run it locally using Docker — no cloud account needed. For this notebook, we're using Qdrant Cloud to make getting started easier and avoid Docker setup. Both approaches work identically with Superlinked!

---


In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
import os

# Verify Qdrant settings are loaded
QDRANT_API_KEY = os.getenv("QDRANT__API_KEY")
QDRANT_HOST = os.getenv("QDRANT__HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT__PORT", "6333"))
QDRANT_USE_HTTPS = os.getenv("QDRANT__USE_HTTPS", "false").lower() == "true"

print(f"📍 Qdrant Host: {QDRANT_HOST}")
print(f"🔌 Qdrant Port: {QDRANT_PORT}")
print(f"🔒 Using HTTPS: {QDRANT_USE_HTTPS}")
print(f"🔑 API Key configured: {bool(QDRANT_API_KEY)}")


📍 Qdrant Host: https://2a719e54-cb16-413c-9f8f-22d685a55ced.europe-west3-0.gcp.cloud.qdrant.io
🔌 Qdrant Port: 6333
🔒 Using HTTPS: False
🔑 API Key configured: True


## Example 1: Defining a Schema


The first step in building any Superlinked application is defining your **Schema** — a structured representation of your data that tells Superlinked what fields exist and what types they are.

For our real estate use case, we'll define a `Property` schema with:

- `id`: Unique identifier (required by Superlinked)
- `description`: Free-text description of the property
- `rooms`: Number of rooms (integer)
- `size_sqm`: Size in square meters (integer)
- `city`: City name (string)
- `district`: District/neighborhood name (string)
- `price`: Price in euros (integer)

Let's define it:


In [3]:
from superlinked import framework as sl


class Property(sl.Schema):
    """Schema for real estate properties."""
    id: sl.IdField
    description: sl.String
    rooms: sl.Integer
    size_sqm: sl.Integer
    city: sl.String
    district: sl.String
    price: sl.Integer


# Create an instance of the schema
property_schema = Property()

print("✅ Property schema defined!")
print(f"   Fields: id, description, rooms, size_sqm, city, district, price")


2025-11-24 21:21:26 [debug    ] YAML configuration not available, falling back to dotenv error=KeyError('yaml_config_section key "framework" not found in config.yaml')
2025-11-24 21:21:26 [debug    ] YAML configuration not available, falling back to dotenv error=KeyError('yaml_config_section key "image" not found in config.yaml')
2025-11-24 21:21:26 [debug    ] YAML configuration not available, falling back to dotenv error=KeyError('yaml_config_section key "resource" not found in config.yaml')
2025-11-24 21:21:26 [debug    ] YAML configuration not available, falling back to dotenv error=KeyError('yaml_config_section key "resource" not found in config.yaml')
2025-11-24 21:21:26 [debug    ] YAML configuration not available, falling back to dotenv error=KeyError('yaml_config_section key "resource" not found in config.yaml')


/Users/jesuscopado/Development/realtime-phone-agents-course/.venv/lib/python3.11/site-packages/pydantic_settings/main.py:426: UserWarning: Config key `yaml_file` is set in model_config but will be ignored because no YamlConfigSettingsSource source is configured. To use this config key, add a YamlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  self._settings_warn_unused_config_keys(sources, self.model_config)
/Users/jesuscopado/Development/realtime-phone-agents-course/.venv/lib/python3.11/site-packages/pydantic_settings/main.py:426: UserWarning: Config key `yaml_config_section` is set in model_config but will be ignored because no YamlConfigSettingsSource source is configured. To use this config key, add a YamlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  self._settings_warn_unused_config_keys(sources, self.model_config)


✅ Property schema defined!
   Fields: id, description, rooms, size_sqm, city, district, price


## Example 2: Creating Vector Spaces


Now comes the magic! We'll create **Spaces** that define how different attributes should be embedded into vectors.

For our property search, we'll create three spaces:

1. **TextSimilaritySpace** for `description`: This uses a sentence transformer model to create semantic embeddings. When someone searches for "modern minimalist design", it will match properties described with similar concepts even if the exact words differ.

2. **NumberSpace** for `size_sqm` (MAXIMUM mode): Larger is better. A search for "spacious" will favor larger apartments.

3. **NumberSpace** for `price` (MINIMUM mode): Lower is better. A search mentioning "affordable" will favor cheaper options.

The `mode` parameter is crucial:
- `Mode.MAXIMUM`: Higher values are preferred (great for size, ratings, popularity)
- `Mode.MINIMUM`: Lower values are preferred (great for price, distance)
- `Mode.SIMILAR`: Values closer to a target are preferred


In [4]:
# Embedding model for text similarity
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# TextSimilaritySpace for semantic understanding of descriptions
description_space = sl.TextSimilaritySpace(
    text=property_schema.description,
    model=EMBEDDING_MODEL
)

# NumberSpace for size - MAXIMUM mode means larger is better
size_space = sl.NumberSpace(
    number=property_schema.size_sqm,
    min_value=20,    # Smallest reasonable apartment
    max_value=200,   # Largest reasonable apartment
    mode=sl.Mode.MAXIMUM
)

# NumberSpace for price - MINIMUM mode means lower is better
price_space = sl.NumberSpace(
    number=property_schema.price,
    min_value=100000,    # Minimum price
    max_value=1000000,   # Maximum price
    mode=sl.Mode.MINIMUM
)

print("✅ Vector spaces created!")
print(f"   📝 Description space: Semantic similarity using {EMBEDDING_MODEL}")
print(f"   📐 Size space: NumberSpace with MAXIMUM mode (larger = better)")
print(f"   💰 Price space: NumberSpace with MINIMUM mode (lower = better)")


`torch_dtype` is deprecated! Use `dtype` instead!
Model download issue, forcing re-download.


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Vector spaces created!
   📝 Description space: Semantic similarity using sentence-transformers/all-MiniLM-L6-v2
   📐 Size space: NumberSpace with MAXIMUM mode (larger = better)
   💰 Price space: NumberSpace with MINIMUM mode (lower = better)


## Example 3: Building an Index


The **Index** is where everything comes together. It combines all your spaces into a single searchable structure.

We'll also include additional fields that we want to be able to filter on (like `city`, `district`, `rooms`) but that don't need their own vector space — they'll be used as hard filters.

This is a key design decision:
- **Spaces**: For attributes that should influence *similarity scoring* (fuzzy matching)
- **Fields**: For attributes used in *exact filtering* (hard constraints)


In [5]:
# Create the index combining all spaces
property_index = sl.Index(
    spaces=[description_space, size_space, price_space],
    fields=[
        property_schema.rooms,
        property_schema.size_sqm,
        property_schema.price,
        property_schema.city,
        property_schema.district,
    ],
)

print("✅ Index created!")
print(f"   🔍 Searchable spaces: description, size, price")
print(f"   🏷️  Filterable fields: rooms, size_sqm, price, city, district")


✅ Index created!
   🔍 Searchable spaces: description, size, price
   🏷️  Filterable fields: rooms, size_sqm, price, city, district


## Example 4: Defining Parameterized Queries


Now we define **Queries** — templates for how we'll search our index. The power of Superlinked queries lies in their parameterization:

- **Weights**: How much each space contributes to the final score (adjustable at query time!)
- **Filters**: Hard constraints on field values
- **Similar**: The search query itself

This is incredibly powerful for a voice agent! When a user says *"I want something affordable"*, we can increase the `price_weight`. When they say *"I need lots of space"*, we increase the `size_weight`. The query adapts to user intent!


In [6]:
# Define the semantic search query with parameterized weights and filters
search_query = (
    sl.Query(
        property_index,
        weights={
            description_space: sl.Param("description_weight"),
            size_space: sl.Param("size_weight"),
            price_space: sl.Param("price_weight"),
        },
    )
    .find(property_schema)
    .similar(
        description_space,
        sl.Param(
            "description_query",
            description="The user's natural language query for property search.",
        ),
    )
    # Filters - these are hard constraints
    .filter(property_schema.city == sl.Param("city"))
    .filter(property_schema.district == sl.Param("district"))
    .filter(property_schema.rooms >= sl.Param("min_rooms"))
    .filter(property_schema.rooms <= sl.Param("max_rooms"))
    .filter(property_schema.size_sqm >= sl.Param("min_size"))
    .filter(property_schema.size_sqm <= sl.Param("max_size"))
    .filter(property_schema.price >= sl.Param("min_price"))
    .filter(property_schema.price <= sl.Param("max_price"))
    .limit(sl.Param("limit"))
    .select_all()
)

print("✅ Search query defined!")
print("   Parameters available:")
print("   - description_query: Natural language search")
print("   - description_weight, size_weight, price_weight: Space weights")
print("   - city, district: Location filters")
print("   - min_rooms, max_rooms: Room count range")
print("   - min_size, max_size: Size range in sqm")
print("   - min_price, max_price: Price range")
print("   - limit: Max results to return")


✅ Search query defined!
   Parameters available:
   - description_query: Natural language search
   - description_weight, size_weight, price_weight: Space weights
   - city, district: Location filters
   - min_rooms, max_rooms: Room count range
   - min_size, max_size: Size range in sqm
   - min_price, max_price: Price range
   - limit: Max results to return


## Example 5: Setting Up the Executor with Qdrant


Superlinked needs an **Executor** to actually run queries. We have two options:

1. **InMemoryExecutor**: Great for development and testing, but data is lost when the process ends
2. **RestExecutor with QdrantVectorDatabase**: Production-ready, persistent storage

We'll use Qdrant — a high-performance vector database — to store our embeddings persistently. You can get a free cluster at [cloud.qdrant.io](https://cloud.qdrant.io)!

Let's set up our executor:


In [7]:
# Build the Qdrant URL
protocol = "https" if QDRANT_USE_HTTPS else "http"
qdrant_url = f"{protocol}://{QDRANT_HOST}:{QDRANT_PORT}"

print(f"🔗 Connecting to Qdrant at: {qdrant_url}")

# Create the Qdrant vector database connection
vector_db = sl.QdrantVectorDatabase(
    url=qdrant_url,
    api_key=QDRANT_API_KEY,
    default_query_limit=10,
)

print("✅ Qdrant vector database configured!")


🔗 Connecting to Qdrant at: http://https://2a719e54-cb16-413c-9f8f-22d685a55ced.europe-west3-0.gcp.cloud.qdrant.io:6333


AttributeError: module 'superlinked.framework' has no attribute 'QdrantVectorDatabase'

In [ ]:
# Create a REST source for data ingestion
source = sl.RestSource(property_schema)

# Create a REST descriptor and query for our search
search_descriptor = sl.RestDescriptor(query_path="/search")
rest_query = sl.RestQuery(search_descriptor, search_query)

# Create the RestExecutor with Qdrant
executor = sl.RestExecutor(
    sources=[source],
    indices=[property_index],
    queries=[rest_query],
    vector_database=vector_db,
)

# Run the executor to get the app
app = executor.run()

print("✅ Executor running with Qdrant backend!")
print("   Ready to ingest data and execute queries.")


## Example 6: Ingesting Property Data


Now let's create some mock property data and ingest it into our system. In a real application, this data would come from your database or API.

When we ingest data, Superlinked automatically:
1. Generates embeddings for each space (text similarity, price encoding, size encoding)
2. Concatenates them into a single vector
3. Stores them in Qdrant with the original field values as metadata


In [ ]:
# Sample property data
mock_properties = [
    {
        "id": "1",
        "description": "Beautiful modern apartment with panoramic city views. Open-plan living with floor-to-ceiling windows, designer kitchen, and a spacious balcony perfect for entertaining.",
        "rooms": 3,
        "size_sqm": 120,
        "city": "Barcelona",
        "district": "Eixample",
        "price": 450000
    },
    {
        "id": "2",
        "description": "Cozy studio in the heart of the Gothic Quarter. Recently renovated with exposed brick walls, modern amenities, and original architectural details preserved.",
        "rooms": 1,
        "size_sqm": 45,
        "city": "Barcelona",
        "district": "Ciutat Vella",
        "price": 180000
    },
    {
        "id": "3",
        "description": "Spacious family home with private garden. Four bedrooms, two bathrooms, large living area, and a fully equipped kitchen. Quiet residential neighborhood.",
        "rooms": 4,
        "size_sqm": 180,
        "city": "Barcelona",
        "district": "Sarria",
        "price": 750000
    },
    {
        "id": "4",
        "description": "Minimalist loft-style apartment in a converted warehouse. High ceilings, industrial chic design, open space concept with mezzanine bedroom.",
        "rooms": 2,
        "size_sqm": 95,
        "city": "Barcelona",
        "district": "Poblenou",
        "price": 320000
    },
    {
        "id": "5",
        "description": "Charming penthouse with private rooftop terrace. Mediterranean style with terracotta floors, whitewashed walls, and breathtaking sea views.",
        "rooms": 2,
        "size_sqm": 85,
        "city": "Barcelona",
        "district": "Barceloneta",
        "price": 395000
    },
    {
        "id": "6",
        "description": "Affordable starter apartment ideal for young professionals. Compact but efficient layout, modern finishes, close to public transportation.",
        "rooms": 1,
        "size_sqm": 38,
        "city": "Barcelona",
        "district": "Sant Andreu",
        "price": 145000
    },
    {
        "id": "7",
        "description": "Luxurious duplex in exclusive neighborhood. Premium finishes throughout, smart home technology, private parking, and concierge service.",
        "rooms": 5,
        "size_sqm": 200,
        "city": "Barcelona",
        "district": "Pedralbes",
        "price": 950000
    },
    {
        "id": "8",
        "description": "Bright corner apartment with excellent natural light. Two sunny bedrooms, renovated bathroom, and a functional kitchen with dining area.",
        "rooms": 2,
        "size_sqm": 72,
        "city": "Barcelona",
        "district": "Gracia",
        "price": 285000
    },
]

print(f"📦 Prepared {len(mock_properties)} properties for ingestion")


In [ ]:
# Ingest the properties into the vector store
source.put(mock_properties)

print(f"✅ Successfully ingested {len(mock_properties)} properties!")
print("   Embeddings generated and stored in Qdrant.")


## Example 7: Running Semantic Searches


Now for the exciting part — let's run some searches! We'll demonstrate how the weights affect results and how natural language queries work.

We'll create a helper function that prints results nicely:


In [ ]:
def display_results(result, query_description: str):
    """Display search results in a readable format."""
    entries = result.model_dump()["entries"]
    
    print(f"\n🔍 Query: {query_description}")
    print(f"   Found {len(entries)} results:\n")
    
    for i, entry in enumerate(entries, 1):
        fields = entry["fields"]
        print(f"   {i}. {fields['district']}, {fields['city']}")
        print(f"      💰 €{fields['price']:,} | 🏠 {fields['rooms']} rooms | 📐 {fields['size_sqm']} sqm")
        print(f"      📝 {fields['description'][:80]}...")
        print()


### Search 1: Pure Semantic Search

Let's search for "modern minimalist design" with high weight on description similarity:


In [ ]:
result = app.query(
    search_query,
    description_query="modern minimalist design",
    description_weight=0.9,
    size_weight=0.05,
    price_weight=0.05,
    limit=3,
)

display_results(result, "modern minimalist design (semantic focus)")


### Search 2: Affordable + Semantic

Now let's search for something affordable. We'll increase the price weight:


In [ ]:
result = app.query(
    search_query,
    description_query="cozy apartment for young professional",
    description_weight=0.5,
    size_weight=0.1,
    price_weight=0.4,  # High weight on price!
    limit=3,
)

display_results(result, "cozy apartment (price-conscious)")


### Search 3: Spacious + Luxury

For someone who wants space and luxury, we'll prioritize size:


In [ ]:
result = app.query(
    search_query,
    description_query="spacious luxury family home",
    description_weight=0.4,
    size_weight=0.5,  # High weight on size!
    price_weight=0.1,
    limit=3,
)

display_results(result, "spacious luxury home (size-focused)")


### Search 4: With Location Filter

Let's add a hard filter for a specific district:


In [ ]:
result = app.query(
    search_query,
    description_query="nice apartment with good views",
    description_weight=0.7,
    size_weight=0.15,
    price_weight=0.15,
    city="Barcelona",
    district="Eixample",  # Hard filter!
    limit=3,
)

display_results(result, "nice apartment in Eixample (filtered)")


Notice how the results change based on the weights we set! This is the power of multi-attribute search — you can tune the relevance dynamically based on user intent.


### Search 5: Room Count Filter

Search for properties with at least 3 rooms:


In [ ]:
result = app.query(
    search_query,
    description_query="family home with garden or outdoor space",
    description_weight=0.6,
    size_weight=0.2,
    price_weight=0.2,
    min_rooms=3,  # At least 3 rooms
    limit=3,
)

display_results(result, "family home with 3+ rooms")


## Example 8: Building a Property Search Service


Now let's encapsulate everything into a reusable service class. This is exactly what we'll use in our voice agent to power the property search tool!

The service will:
1. Handle connection to Qdrant (with fallback to in-memory)
2. Provide a clean interface for searching
3. Convert results to a format suitable for our agent


In [ ]:
from typing import Any


class PropertySearchService:
    """Service for property search using Superlinked."""

    def __init__(
        self,
        qdrant_host: str,
        qdrant_port: int,
        qdrant_api_key: str | None = None,
        use_https: bool = True,
    ):
        """Initialize the property search service."""
        self.qdrant_host = qdrant_host
        self.qdrant_port = qdrant_port
        self.qdrant_api_key = qdrant_api_key
        self.use_https = use_https
        
        # Setup the Superlinked app
        self._setup_app()

    def _setup_app(self):
        """Setup the Superlinked application with Qdrant."""
        # Build Qdrant URL
        protocol = "https" if self.use_https else "http"
        qdrant_url = f"{protocol}://{self.qdrant_host}:{self.qdrant_port}"
        
        # Create Qdrant connection
        vector_db = sl.QdrantVectorDatabase(
            url=qdrant_url,
            api_key=self.qdrant_api_key,
            default_query_limit=10,
        )
        
        # Create source and executor
        self.source = sl.RestSource(property_schema)
        search_desc = sl.RestDescriptor(query_path="/search")
        rest_query = sl.RestQuery(search_desc, search_query)
        
        self.executor = sl.RestExecutor(
            sources=[self.source],
            indices=[property_index],
            queries=[rest_query],
            vector_database=vector_db,
        )
        
        self.app = self.executor.run()
        print(f"✅ PropertySearchService connected to Qdrant at {qdrant_url}")

    def _result_to_properties(self, result) -> list[dict[str, Any]]:
        """Convert QueryResult to clean property dicts."""
        entries = result.model_dump()["entries"]
        return [{**entry["fields"], "id": int(entry["id"])} for entry in entries]

    def search_properties(
        self,
        description_query: str,
        limit: int = 3,
        description_weight: float = 0.7,
        size_weight: float = 0.15,
        price_weight: float = 0.15,
        city: str | None = None,
        district: str | None = None,
        min_rooms: int | None = None,
        max_rooms: int | None = None,
        min_price: int | None = None,
        max_price: int | None = None,
    ) -> list[dict[str, Any]]:
        """Search for properties using semantic similarity."""
        # Build query parameters
        query_params = {
            "description_query": description_query,
            "limit": limit,
            "description_weight": description_weight,
            "size_weight": size_weight,
            "price_weight": price_weight,
        }
        
        # Add optional filters
        if city:
            query_params["city"] = city
        if district:
            query_params["district"] = district
        if min_rooms is not None:
            query_params["min_rooms"] = min_rooms
        if max_rooms is not None:
            query_params["max_rooms"] = max_rooms
        if min_price is not None:
            query_params["min_price"] = min_price
        if max_price is not None:
            query_params["max_price"] = max_price
        
        # Execute search
        result = self.app.query(search_query, **query_params)
        
        return self._result_to_properties(result)

    def format_for_agent(self, properties: list[dict]) -> str:
        """Format properties for voice agent response."""
        if not properties:
            return "I couldn't find any properties matching your criteria."
        
        lines = [f"I found {len(properties)} properties for you:\n"]
        
        for i, prop in enumerate(properties, 1):
            lines.append(
                f"{i}. A {prop['rooms']}-room apartment in {prop['district']} "
                f"with {prop['size_sqm']} square meters, priced at {prop['price']:,} euros. "
                f"{prop['description'][:100]}..."
            )
        
        return "\n".join(lines)


print("✅ PropertySearchService class defined!")


Let's test our service:


In [ ]:
# Create a new service instance
service = PropertySearchService(
    qdrant_host=QDRANT_HOST,
    qdrant_port=QDRANT_PORT,
    qdrant_api_key=QDRANT_API_KEY,
    use_https=QDRANT_USE_HTTPS,
)

# Search for properties
properties = service.search_properties(
    description_query="I want a modern apartment with good views",
    limit=3,
)

# Format for agent response
agent_response = service.format_for_agent(properties)
print("\n🎙️ Agent Response:")
print(agent_response)


## Example 9: Creating a LangChain Tool


Finally, let's wrap our service in a LangChain tool that our voice agent can use. This is the bridge between the conversational AI and our Superlinked-powered search!

The tool will accept natural language queries and return formatted results that the agent can speak to the user.


In [ ]:
from langchain.tools import tool


# Create a global service instance (in production, this would be injected)
_search_service = PropertySearchService(
    qdrant_host=QDRANT_HOST,
    qdrant_port=QDRANT_PORT,
    qdrant_api_key=QDRANT_API_KEY,
    use_https=QDRANT_USE_HTTPS,
)


@tool
def search_properties_tool(
    query: str,
    city: str | None = None,
    district: str | None = None,
    min_rooms: int | None = None,
    max_price: int | None = None,
) -> str:
    """
    Search for real estate properties using natural language.
    
    Use this tool when a user asks about apartments or properties.
    The query should describe what the user is looking for.
    
    Args:
        query: Natural language description of desired property
        city: Filter by city name (optional)
        district: Filter by district/neighborhood (optional)
        min_rooms: Minimum number of rooms (optional)
        max_price: Maximum price in euros (optional)
    
    Returns:
        Formatted string with property results for voice response
    """
    properties = _search_service.search_properties(
        description_query=query,
        limit=3,
        city=city,
        district=district,
        min_rooms=min_rooms,
        max_price=max_price,
    )
    
    return _search_service.format_for_agent(properties)


print("✅ LangChain tool created!")
print(f"   Tool name: {search_properties_tool.name}")
print(f"   Description: {search_properties_tool.description[:80]}...")


Let's test the tool directly:


In [ ]:
# Test the tool
result = search_properties_tool.invoke({
    "query": "affordable cozy apartment for a single person",
    "max_price": 300000,
})

print("🎙️ Tool Response:")
print(result)


## Example 10: Integration with a Voice Agent


Now let's bring it all together! We'll create a simple agent that uses our property search tool. This is exactly how we'll integrate it with FastRTC in the next lesson.

The agent will:
1. Receive natural language queries from the user
2. Decide when to use the search tool
3. Format results for spoken response


In [ ]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

from realtime_phone_agents.config import settings

system_prompt = """
Your name is Sofia, and you work for The Neural Maze Real Estate company.
Your task is to help users find their perfect property using the search_properties_tool tool.

When a user describes what they're looking for, use the search tool to find matching properties.
Extract any specific requirements they mention (budget, rooms, location) and pass them to the tool.

Be conversational and helpful. Don't use emojis or asterisks since this is a phone call.
Keep responses concise but informative.
"""

# Create the LLM
llm = ChatGroq(model=settings.groq.model, api_key=settings.groq.api_key)

# Create the agent with our property search tool
property_agent = create_agent(
    llm,
    checkpointer=InMemorySaver(),
    system_prompt=system_prompt,
    tools=[search_properties_tool],
)

print("✅ Property search agent created!")


In [ ]:
# Test the agent
response = property_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hi! I'm looking for a modern apartment in Barcelona, "
                           "preferably with at least 2 rooms and under 400 thousand euros."
            }
        ]
    },
    {"configurable": {"thread_id": "test-session"}},
)

print("🎙️ Agent Response:")
print(response["messages"][-1].content)


## Key Takeaways


In this lesson, you learned:

1. **Why traditional vector search falls short**: Text-only embeddings can't handle multi-attribute queries that combine semantic meaning with numerical and categorical data.

2. **How Superlinked solves this**: By providing specialized Spaces for different data types and allowing runtime weight adjustment, Superlinked enables truly intelligent multi-attribute search.

3. **The building blocks**:
   - **Schema**: Define your data structure
   - **Spaces**: TextSimilaritySpace, NumberSpace for different attribute types
   - **Index**: Combine spaces and define filterable fields
   - **Query**: Parameterized templates with weights and filters
   - **Executor**: Run queries against Qdrant or in-memory

4. **Integration pattern**: Wrap Superlinked in a service class, then expose it as a LangChain tool for your agent.